# NB1c — Tile → Cell Feature Aggregation → `lucia_cell_features.parquet`

**Env:** `datasci`
**Input:** `lucia_tile_features.parquet` · `lucia_cells.parquet` · `lucia_geometry.parquet`
**Output:** `lucia_cell_features.parquet` (~75 columns per cell after 1c patch)

Pipeline:
1. Load tiles + modelling cohort
2. ~~Region aggregation (5 regions × 7 channels)~~ **DROPPED** (1c patch)
3. Non-uniformity CV scalars (`nu_cv`, `nu_p95p5`, `nu_fraclow`) — 7 ch × 3 = 21 cols
4. ~~Physics log1p transforms~~ **DROPPED** (1c patch)
5. Join geometry scalars (rms_resid, theta_deg, sat_frac_*, low_signal_*)
6. Save (~31 pre-PCA cols) + Spearman heatmap
7. Tile spatial PCA (4 channels × 10 PCs = 40 cols) → update parquet → ~71 cols total

In [1]:
# ── Config & imports ────────────────────────────────────────────────────────
import os, sys, json, time, warnings
import numpy as np
import pandas as pd
import scipy.stats
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')   # headless — no display on UBELIX; savefig only

_LUCIA_ROOT = os.environ.get('LUCIA_ROOT', '/storage/homefs/db98d082/ondemand/LUCIA')
sys.path.insert(0, os.path.join(_LUCIA_ROOT, 'notebooks_scripts'))
import lucia_common as lc

# Channels present in tile features
CHANNELS        = ['el_lo', 'el_hi', 'pl_hi', 'pl_lo', 'rs_map', 'log_el_pl', 'grad_el_hi']
# After 37-feature regen, entropy & skew exist on ALL 7 channels
ENTROPY_SKEW_CH = CHANNELS
RAW_CH          = ['el_lo', 'el_hi', 'pl_hi', 'pl_lo']  # for reference only

os.makedirs(lc.FIGURES, exist_ok=True)
print(f"LUCIA_ROOT: {_LUCIA_ROOT}")
print(f"Output → {lc.CELL_FEATURES_PARQUET}")

LUCIA_ROOT: /storage/homefs/db98d082/ondemand/LUCIA
Output → /storage/homefs/db98d082/ondemand/LUCIA/data/processed/lucia_cell_features.parquet


In [2]:
# ── Load parquets + tile features ───────────────────────────────────────────
t0 = time.perf_counter()
cells, geom = lc.load_parquets()

# Modelling cohort: all cells with valid IV + geometry (paired=False keeps ambiguous)
cohort_mask  = lc.modeling_mask(cells, geom, paired=False)
cohort_names = set(cohort_mask[cohort_mask].index)
print(f"Modelling cohort: {len(cohort_names):,} cells")
assert len(cohort_names) > 5000, \
    f"Cohort too small ({len(cohort_names)}): stale lucia_cells.parquet or wrong LUCIA_ROOT?"

# Read only the columns we need from the 205 MB tile parquet
mean_cols = [f'mean_{ch}' for ch in CHANNELS]
std_cols  = [f'std_{ch}'  for ch in CHANNELS]
ent_cols  = [f'entropy_{ch}' for ch in ENTROPY_SKEW_CH]
skw_cols  = [f'skew_{ch}'    for ch in ENTROPY_SKEW_CH]
needed    = ['is_border', 'half', 'active_frac'] + mean_cols + std_cols + ent_cols + skw_cols
# KeyError here means the tile parquet is the old (<37-feature) version — re-run NB1b.

print("Loading tile parquet (selected cols)...", end=" ", flush=True)
tf = pd.read_parquet(lc.TILE_PARQUET, columns=needed)
print(f"{time.perf_counter()-t0:.1f}s  {len(tf):,} rows  {len(tf.columns)} columns")
print(f"  Source: {lc.TILE_PARQUET}")

# Filter to modelling cohort
lvl0 = tf.index.get_level_values(0)
tf   = tf[lvl0.isin(cohort_names)].copy()
lvl0 = tf.index.get_level_values(0)
print(f"After cohort filter: {len(tf):,} rows  {lvl0.nunique():,} cells")

Modelling cohort: 10,427 cells
Loading tile parquet (selected cols)... 2.5s  1,699,704 rows  31 columns
  Source: /storage/homefs/db98d082/ondemand/LUCIA/data/processed/lucia_tile_features.parquet
After cohort filter: 1,689,174 rows  10,427 cells


In [3]:
# ── Region masks DROPPED (1c patch) ──────────────────────────────────────────
# 5-region aggregates (whole/internal/border/left/right, ~110 cols) removed.
# tf still in scope for the nu_cv computation below.
print("Region aggregates dropped — tf available for nu_cv cell.")

Region aggregates dropped — tf available for nu_cv cell.


In [4]:
# ── Aggregation function DROPPED (1c patch) ───────────────────────────────────
# Region aggregation removed; nu_cv computed directly in the next cell.
print("Aggregation function dropped.")

Aggregation function dropped.


In [5]:
# ── feats initialised (region aggregation DROPPED, 1c patch) ─────────────────
# nu_cv + geometry + tile-PCA fill feats in the cells below.
feats = pd.DataFrame(index=sorted(cohort_names))
print(f"feats initialised empty: {feats.shape}")

feats initialised empty: (10427, 0)


In [6]:
# ── Non-uniformity CV (between-tile spread, brightness-invariant) ─────────────
# nu_cv = std(tile_means) / mean(tile_means) — coefficient of variation.
# Brightness-invariant (unlike raw std): same spatial non-uniformity at different
# absolute intensities gives the same CV.  Physics: CV → local Rs variability → FF.
# Also kept: nu_p95p5 (robust spread) and nu_fraclow (dark-defect tile fraction).

nu = {}
idx_all = tf.index.get_level_values(0)

for ch in CHANNELS:
    m    = tf[f'mean_{ch}']
    grp  = m.groupby(idx_all)

    ch_mean = grp.mean().clip(lower=1e-6)    # avoid /0 for dark channels
    ch_std  = grp.std()
    q05     = grp.quantile(0.05)
    q25     = grp.quantile(0.25)
    q75     = grp.quantile(0.75)

    nu[f'{ch}_nu_cv']      = ch_std / ch_mean          # coefficient of variation
    nu[f'{ch}_nu_p95p5']   = grp.quantile(0.95) - q05  # robust spread

    thr_exp = (q05 + 0.5 * (q75 - q25)).reindex(idx_all).values
    below   = (m.values < thr_exp).astype(np.float32)
    nu[f'{ch}_nu_fraclow'] = pd.Series(below, index=tf.index).groupby(idx_all).mean()

df_nu = pd.DataFrame(nu)
print(f"Non-uniformity features: {df_nu.shape[1]} cols  {df_nu.shape[0]:,} cells")
feats = feats.join(df_nu, how='left')

Non-uniformity features: 21 cols  10,427 cells


In [7]:
# ── Log1p region transforms DROPPED (1c patch) ───────────────────────────────
# These were log1p of region-aggregated means; with regions gone they go too.
print("Log1p region transforms dropped.")

Log1p region transforms dropped.


In [8]:
# ── Join geometry scalars ─────────────────────────────────────────────────────
GEOM_COLS = (
    ['rms_resid', 'theta_deg']
    + [c for c in geom.columns if c.startswith('sat_frac_')]
    + [c for c in geom.columns if c.startswith('low_signal_')]
)
geom_sc = geom[GEOM_COLS].copy()
for col in geom_sc.columns:
    if geom_sc[col].dtype == bool:
        geom_sc[col] = geom_sc[col].astype(np.float32)

feats = feats.join(geom_sc, how='left')
print(f"After geometry join: {feats.shape}")


After geometry join: (10427, 31)


In [9]:
# ── Assemble, validate NaN, save ──────────────────────────────────────────────
feats.index.name = 'cell_name'
feats_out = feats.reindex(sorted(cohort_names)).dropna(how='all')

nan_total = feats_out.isna().sum().sum()
print(f"Shape: {feats_out.shape}  NaN total: {nan_total}")
if nan_total > 0:
    top_nan = feats_out.isna().sum().sort_values(ascending=False).head(10)
    print("Top NaN columns:\n", top_nan[top_nan > 0])

feats_out.to_parquet(lc.CELL_FEATURES_PARQUET, engine='pyarrow')
sz = os.path.getsize(lc.CELL_FEATURES_PARQUET) / 1e6
print(f"Saved: {lc.CELL_FEATURES_PARQUET}  ({sz:.1f} MB)")

print("\n── Feature groups (pre-PCA; tile-PCA cell adds 40 more) ──")
nu_cv_cols     = [c for c in feats_out.columns if c.endswith('_nu_cv')]
nu_p95p5_cols  = [c for c in feats_out.columns if c.endswith('_nu_p95p5')]
nu_fraclow_cols= [c for c in feats_out.columns if c.endswith('_nu_fraclow')]
geom_cols      = [c for c in feats_out.columns
                  if not any(c.endswith(s) for s in ('_nu_cv','_nu_p95p5','_nu_fraclow'))]
for name, cols in [
    ('nu_cv (7ch)', nu_cv_cols),
    ('nu_p95p5',    nu_p95p5_cols),
    ('nu_fraclow',  nu_fraclow_cols),
    ('geometry',    geom_cols),
]:
    print(f"  {name:20}: {len(cols):>3} cols")
print(f"  {'TOTAL (pre-PCA)':20}: {feats_out.shape[1]:>3} cols")
print(f"  {'Expected final':20}: ~{feats_out.shape[1] + 40:>2} cols (+ 40 tile PCA)")

Shape: (10427, 31)  NaN total: 0
Saved: /storage/homefs/db98d082/ondemand/LUCIA/data/processed/lucia_cell_features.parquet  (1.7 MB)

── Feature groups (pre-PCA; tile-PCA cell adds 40 more) ──
  nu_cv (7ch)         :   7 cols
  nu_p95p5            :   7 cols
  nu_fraclow          :   7 cols
  geometry            :  10 cols
  TOTAL (pre-PCA)     :  31 cols
  Expected final      : ~71 cols (+ 40 tile PCA)


In [10]:
# ── Spearman correlation heatmap: top-20 features vs IV ──────────────────────
import matplotlib.pyplot as plt

IV_TARGETS = ['Voc', 'Isc', 'FF', 'Pmax']
cells_k    = cells.loc[feats_out.index, IV_TARGETS].astype(float)
iv_valid   = cells_k.dropna().index

rho_rows = {}
for feat in feats_out.columns:
    vals   = feats_out[feat].reindex(iv_valid).dropna()
    common = vals.index.intersection(iv_valid)
    if len(common) < 100:
        continue
    rhos = [scipy.stats.spearmanr(vals.loc[common], cells_k.loc[common, t])[0]
            for t in IV_TARGETS]
    rho_rows[feat] = rhos

rho_df  = pd.DataFrame(rho_rows, index=IV_TARGETS).T
top20   = rho_df.assign(max_abs=rho_df.abs().max(axis=1)).nlargest(20, 'max_abs').drop(columns='max_abs')

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(top20.T.values, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20.index, rotation=90, fontsize=7)
ax.set_yticks(range(len(IV_TARGETS)))
ax.set_yticklabels(IV_TARGETS, fontsize=11)
for i, feat in enumerate(top20.index):
    for j, tgt in enumerate(IV_TARGETS):
        ax.text(i, j, f"{top20.loc[feat, tgt]:.2f}", ha='center', va='center', fontsize=6)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02)
ax.set_title("Top-20 features by max |ρ| vs IV (Spearman)", fontsize=12)
plt.tight_layout()
out_png = f"{lc.FIGURES}/nb1c_feature_iv_heatmap.png"
fig.savefig(out_png, dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved → {out_png}")
print("\nTop-5 features for each IV target:")
for tgt in IV_TARGETS:
    top5 = rho_df[tgt].abs().nlargest(5)
    print(f"  {tgt}: {list(zip(top5.index, top5.values.round(3)))}")


/scratch/local/7444702/ipykernel_3542489/672965448.py:14: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rhos = [scipy.stats.spearmanr(vals.loc[common], cells_k.loc[common, t])[0]


Saved → /storage/homefs/db98d082/ondemand/LUCIA/outputs/figures/nb1c_feature_iv_heatmap.png

Top-5 features for each IV target:
  Voc: [('el_lo_nu_p95p5', 0.573), ('rs_map_nu_cv', 0.501), ('pl_lo_nu_cv', 0.405), ('el_hi_nu_cv', 0.372), ('sat_frac_pl_hi', 0.37)]
  Isc: [('rs_map_nu_cv', 0.417), ('pl_lo_nu_cv', 0.342), ('el_hi_nu_cv', 0.321), ('log_el_pl_nu_cv', 0.299), ('pl_hi_nu_cv', 0.296)]
  FF: [('rs_map_nu_cv', 0.724), ('el_hi_nu_cv', 0.678), ('log_el_pl_nu_p95p5', 0.672), ('el_lo_nu_cv', 0.65), ('log_el_pl_nu_cv', 0.634)]
  Pmax: [('rs_map_nu_cv', 0.721), ('el_hi_nu_cv', 0.63), ('log_el_pl_nu_p95p5', 0.628), ('log_el_pl_nu_cv', 0.603), ('el_lo_nu_cv', 0.575)]


In [11]:
# ── Tile spatial PCA features ─────────────────────────────────────────────────
# Region aggregates collapse 162 tiles → 5 coarse buckets, discarding spatial
# structure. PCA of the per-cell tile-mean matrix retains that structure as
# compact features: PC1 ≈ overall brightness variation; PC2 ≈ horizontal
# gradient; PC3 ≈ crack / finger pattern; etc.
#
# For each channel: (N_cells × N_tiles) → StandardScaler → PCA → top-k projections.
# The loadings (162-vectors) are saved for spatial visualisation in the next cell.

import pickle
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PCA_CHANNELS = ['el_hi', 'pl_hi', 'rs_map', 'log_el_pl']
N_PCS        = 10

pca_artifacts = {}
all_pca_feats = {}

for ch in tqdm(PCA_CHANNELS, desc="Tile PCA"):
    col = f'mean_{ch}'

    # (N_cells × N_tiles) via unstack — fast pandas pivot
    # tf is already cohort-filtered, so no NaN cell_names here
    mat_df = tf[col].unstack(level='tile_id').astype(np.float32)

    # Drop NaN-named tile_id column (artifact of the batch start NaN row in parquet)
    # and any tile_id present in <5% of cells (sparse boundary tiles)
    valid_cols = mat_df.columns.notna() & (mat_df.isna().mean(axis=0) < 0.05)
    mat_df     = mat_df.loc[:, valid_cols]

    # Fill remaining NaN (sparse boundary tiles that passed the 5% threshold) with column median
    mat_df = mat_df.fillna(mat_df.median(axis=0))
    mat    = mat_df.values   # (N × n_tiles)

    # Per-tile standardize: removes tile-mean brightness, keeps relative variation
    scaler = StandardScaler()
    mat_sc = scaler.fit_transform(mat)

    # PCA
    pca  = PCA(n_components=N_PCS, random_state=42)
    proj = pca.fit_transform(mat_sc)   # (N × N_PCS)

    cum_var = pca.explained_variance_ratio_.cumsum()
    print(f"  {ch:15}: {mat.shape[1]} tiles,  top-{N_PCS} PCs = {cum_var[-1]*100:.1f}% variance")

    for k in range(N_PCS):
        all_pca_feats[f'{ch}_pc{k+1:02d}'] = pd.Series(proj[:, k], index=mat_df.index)

    pca_artifacts[ch] = {
        'pca':       pca,
        'scaler':    scaler,
        'tile_ids':  list(mat_df.columns),          # tile_id order used in fitting
        'var_ratio': pca.explained_variance_ratio_.tolist(),
    }

df_pca = pd.DataFrame(all_pca_feats)
print(f"\nTile PCA features: {df_pca.shape[1]} cols  ({len(PCA_CHANNELS)} ch × {N_PCS} PCs)")

# Save PCA artifacts so NB2 can apply same transform at inference time
artifact_path = f"{lc.PROCESSED}/tile_pca_artifacts.pkl"
with open(artifact_path, 'wb') as f:
    pickle.dump(pca_artifacts, f)
print(f"PCA artifacts → {artifact_path}")

# Append PCA features to the existing cell features parquet
feats_out = pd.read_parquet(lc.CELL_FEATURES_PARQUET)
feats_out = feats_out.join(df_pca, how='left')
feats_out.to_parquet(lc.CELL_FEATURES_PARQUET, engine='pyarrow')
print(f"Updated parquet: {feats_out.shape}  (+{df_pca.shape[1]} PCA cols)")

# Quick IV correlation check
IV_TARGETS = ['Voc', 'Isc', 'FF', 'Pmax']
cells_k    = cells.loc[feats_out.index, IV_TARGETS].astype(float).dropna()
rho_pca    = {}
for feat in df_pca.columns:
    vals   = df_pca[feat].reindex(cells_k.index).dropna()
    common = vals.index.intersection(cells_k.index)
    if len(common) < 100:
        continue
    rho_pca[feat] = [scipy.stats.spearmanr(vals.loc[common],
                     cells_k.loc[common, t])[0] for t in IV_TARGETS]

rho_pca_df = pd.DataFrame(rho_pca, index=IV_TARGETS).T
top10 = rho_pca_df.assign(mx=rho_pca_df.abs().max(axis=1)).nlargest(10,'mx').drop(columns='mx')
print(f"\nTop-10 tile PCA features by max |ρ| with IV:")
print(top10.round(3).to_string())

Tile PCA:  50%|█████     | 2/4 [00:00<00:00,  4.81it/s]

  el_hi          : 162 tiles,  top-10 PCs = 96.8% variance
  pl_hi          : 162 tiles,  top-10 PCs = 97.8% variance
  rs_map         : 162 tiles,  top-10 PCs = 97.4% variance


Tile PCA: 100%|██████████| 4/4 [00:00<00:00,  5.98it/s]


  log_el_pl      : 162 tiles,  top-10 PCs = 98.2% variance

Tile PCA features: 40 cols  (4 ch × 10 PCs)
PCA artifacts → /storage/homefs/db98d082/ondemand/LUCIA/data/processed/tile_pca_artifacts.pkl
Updated parquet: (10427, 71)  (+40 PCA cols)

Top-10 tile PCA features by max |ρ| with IV:
                  Voc    Isc     FF   Pmax
el_hi_pc01      0.898  0.589  0.509  0.671
pl_hi_pc01      0.825  0.561  0.392  0.556
rs_map_pc01     0.614  0.527  0.323  0.473
log_el_pl_pc01  0.520  0.348  0.546  0.589
log_el_pl_pc03  0.210  0.179  0.472  0.421
rs_map_pc03     0.156  0.182  0.396  0.351
el_hi_pc03      0.087  0.144  0.325  0.278
log_el_pl_pc07  0.147  0.043  0.324  0.265
rs_map_pc07     0.137  0.068  0.264  0.219
el_hi_pc07      0.083  0.012  0.257  0.183


  log_el_pl      : 162 tiles,  top-10 PCs = 98.2% variance

Tile PCA features: 40 cols  (4 ch × 10 PCs)
PCA artifacts → /home/derk/DataScience/CAS_projects/LUCIA/data/processed/tile_pca_artifacts.pkl


Updated parquet: (10427, 71)  (+40 PCA cols)



Top-10 tile PCA features by max |ρ| with IV:
                  Voc    Isc     FF   Pmax
el_hi_pc01      0.903  0.597  0.534  0.694
pl_hi_pc01      0.825  0.563  0.395  0.558
log_el_pl_pc01  0.518  0.345  0.601  0.626
rs_map_pc01     0.621  0.536  0.352  0.499
log_el_pl_pc03  0.213  0.179  0.453  0.406
log_el_pl_pc07  0.160  0.074  0.340  0.287
rs_map_pc03     0.136  0.158  0.330  0.291
el_hi_pc03      0.050  0.113  0.253  0.209
rs_map_pc07     0.131  0.080  0.237  0.201
el_hi_pc02      0.039  0.052  0.227  0.175


In [12]:
# ── PC loading spatial visualisation ─────────────────────────────────────────
# Each PC loading is a 162-vector; scatter-plotted at tile (center_x, center_y)
# positions it reveals WHICH spatial pattern the PC captures.
# Red = high loading (tiles that drive this PC positive), Blue = low.

# Get tile positions for one valid cell (skip the NaN artifact at row 0)
tf_pos   = pd.read_parquet(lc.TILE_PARQUET, columns=['center_x', 'center_y'])
# Use first cell from our PCA-fitted matrix (guaranteed valid)
any_cell = df_pca.index[0]
tile_layout = tf_pos.xs(any_cell, level='cell_name')   # tile_id → center_x, center_y

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for row_i, ch in enumerate(['el_hi', 'rs_map']):
    art      = pca_artifacts[ch]
    loadings = art['pca'].components_   # (N_PCS × n_tiles)
    tile_ids = art['tile_ids']

    # Align layout to tile_ids used in PCA (same order as loadings columns)
    pos  = tile_layout.reindex(tile_ids)
    cx   = pos['center_x'].values
    cy   = pos['center_y'].values

    for col_j in range(4):
        ax   = axes[row_i, col_j]
        lv   = loadings[col_j]
        vmax = np.abs(lv).max()

        sc = ax.scatter(cx, -cy, c=lv, cmap='RdBu_r', s=18, vmin=-vmax, vmax=vmax)

        pct  = art['var_ratio'][col_j] * 100
        feat = f'{ch}_pc{col_j+1:02d}'
        if feat in rho_pca_df.index:
            best_tgt = rho_pca_df.loc[feat].abs().idxmax()
            best_rho = rho_pca_df.loc[feat, best_tgt]
            sub = f"best ρ({best_tgt})={best_rho:+.3f}"
        else:
            sub = ""
        ax.set_title(f"{ch}  PC{col_j+1}  ({pct:.1f}%)\n{sub}", fontsize=8)
        ax.set_aspect('equal')
        ax.axis('off')
        plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.01)

plt.suptitle("Tile spatial PC loadings  PC1–PC4\nel_hi (top row) · rs_map (bottom row)\n"
             "Red = tiles that drive PC positive  |  structure = spatial defect modes",
             fontsize=10, y=1.03)
plt.tight_layout()
out_png = f"{lc.FIGURES}/nb1c_tile_pca_loadings.png"
fig.savefig(out_png, dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved → {out_png}")
print("\nFinal parquet feature count:", pd.read_parquet(lc.CELL_FEATURES_PARQUET).shape[1])

Saved → /storage/homefs/db98d082/ondemand/LUCIA/outputs/figures/nb1c_tile_pca_loadings.png

Final parquet feature count: 71
